### Task 1: Conceptual Understanding

#### Q1. What is the difference between "Love" and "love" in NLP?
Answer: 
In NLP, "Love" and "love" are treated as two distinct tokens by default because most tokenizers are case sensitive. This means the model may fail to recognize them as the same word, leading to sparse or inaccurate feature representations, especially being problematic in tasks like sentiment analysis or text classification. Therefore, lowercasing is a basic mandatory preprocessing step.

#### Q2. What happens if stopwords are not removed?
Words like "the", "is", "this","a",etc. appear everywhere but say almost nothing meaningful. If stopword removal is skipped, these words end up dominating the TF-IDF or bag-of-words vectors, drowning out the words that actually matter. Hence, the model ends up chasing noise instead of signal — and tasks like sentiment analysis or text classification are severely affected in terms of accuracy.

#### Q3. Provide two real-world scenarios where removing stopwords can be harmful.
Two real-world scenarios where removing stopwords can be harmful: 
Sentiment analysis with negation — "This is not bad" vs "This is bad." The word "not" is typically flagged as a stopword, but removing it completely reverses the sentiment polarity. Stripping it makes both sentences semantically identical to the model resulting in a critical failure.

Voice assistants and search queries — if a user asks "how do I turn this off?" Once the stopwords are removed, the remaining tokens say "turn off" — losing the intent entirely. Short, conversational queries rely heavily on function words to carry meaning, and blindly removing them breaks the whole interaction.

#### Q4. Explain the difference between stemming and lemmatization.
Both normalize words to a base form, but in different ways.  Stemming is a fast, rule-based process that chops suffixes. "Caring" becomes "car", which is fast but factually wrong. On the other side, Lemmatization is smarter — it uses a proper vocabulary and understands grammar, so "caring" correctly becomes "care", and "worse" becomes "bad". If speed matters more than accuracy, stemming works. But if the task is to actually understand language, lemmatization is a better approach.

### Task 2: Build Advanced Preprocessing Function

In [1]:
import re

import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Laharika\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Laharika\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
def preprocess_text(text):
    # Handling empty or invalid input
    if not text or not isinstance(text, str):
        return [], ""
    
    text = text.lower() # Converting to lowercase
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # To remove URLs
    text = re.sub(r'\S+@\S+', '', text) # To remove email-like patterns
    text = re.sub(r'\d+', '', text) # To remove numbers
    text = re.sub(r'(.)\1{2,}', r'\1', text) # To handle repeated characters
    
    # Tokenizing the text
    tokens = word_tokenize(text)
    
    tokens = [word for word in tokens if word.isalpha()] # Remove non-alphabetic tokens
    tokens = [word for word in tokens if len(word) > 2 or word in ['no', 'not']] # Remove very short tokens except ['no', 'not']
    
    # joining the clean tokens
    cleaned_sentence = " ".join(tokens)

    return tokens, cleaned_sentence

### Task 3: Stress Testing

In [3]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

In [4]:
for text in test_sentences:
    tokens, cleaned_sentence = preprocess_text(text)
    
    print("Original Text   :", text)
    print("Cleaned Tokens  :", tokens)
    print("Cleaned Sentence:", cleaned_sentence)
    print("*" * 60)

Original Text   : Get 100% FREE access now!!!
Cleaned Tokens  : ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
************************************************************
Original Text   : I absolutely looooved this product 😍😍
Cleaned Tokens  : ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence: absolutely loved this product
************************************************************
Original Text   : Worst service ever... 0/10
Cleaned Tokens  : ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
************************************************************
Original Text   : Call me at 9876543210
Cleaned Tokens  : ['call']
Cleaned Sentence: call
************************************************************
Original Text   : This is THE best course!!!
Cleaned Tokens  : ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
************************************************************
Original Text   : Visit https://openai.c

### Task 4: Token Analytics

In [5]:
def token_analytics(text):
    tokens, cleaned_sentence = preprocess_text(text)
    
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_token_length = round(
        sum(len(word) for word in tokens) / total_tokens, 2
    ) if total_tokens > 0 else 0
    
    return {
        "original": text,
        "cleaned_tokens": tokens,
        "cleaned_sentence": cleaned_sentence,
        "total_tokens": total_tokens,
        "unique_tokens": unique_tokens,
        "avg_token_length": avg_token_length
    }

In [6]:
results = [token_analytics(text) for text in test_sentences]

for res in results:
    print("Original Text   :", res["original"])
    print("Tokens          :", res["cleaned_tokens"])
    print("Total Tokens    :", res["total_tokens"])
    print("Unique Tokens   :", res["unique_tokens"])
    print("Avg Token Length:", res["avg_token_length"])
    print("*" * 60)

Original Text   : Get 100% FREE access now!!!
Tokens          : ['get', 'free', 'access', 'now']
Total Tokens    : 4
Unique Tokens   : 4
Avg Token Length: 4.0
************************************************************
Original Text   : I absolutely looooved this product 😍😍
Tokens          : ['absolutely', 'loved', 'this', 'product']
Total Tokens    : 4
Unique Tokens   : 4
Avg Token Length: 6.5
************************************************************
Original Text   : Worst service ever... 0/10
Tokens          : ['worst', 'service', 'ever']
Total Tokens    : 3
Unique Tokens   : 3
Avg Token Length: 5.33
************************************************************
Original Text   : Call me at 9876543210
Tokens          : ['call']
Total Tokens    : 1
Unique Tokens   : 1
Avg Token Length: 4.0
************************************************************
Original Text   : This is THE best course!!!
Tokens          : ['this', 'the', 'best', 'course']
Total Tokens    : 4
Unique Tokens   :

#### Analysis Questions

**1. Which sentence had the most noise?**

“Win $$$ now!!! Limited offer!!!” and “Get 100% FREE access now!!!”

These sentences contain symbols, numbers, and excessive punctuation, resulting in higher noise in the text data.

**2. Which sentence retained the most meaningful tokens after cleaning?**

“I am not happy with this” because:
- Minimal noise initially
- Important semantic word “not” is preserved
- Most tokens remain useful after cleaning

### Task 5: Frequency Analysis

In [7]:
from collections import Counter

In [8]:
# Collecting all tokens from all sentences
all_tokens = []

for sentence in test_sentences:
    tokens,cleaned_sentence = preprocess_text(sentence)
    all_tokens.extend(tokens)

# Creating frequency counter
freq = Counter(all_tokens)

In [9]:
# Top 10 most frequent words
top_10 = freq.most_common(10)

print("Top 10 Most Frequent Words:")
for word, count in top_10:
    print(f"{word}: {count}")

Top 10 Most Frequent Words:
this: 4
now: 3
get: 1
free: 1
access: 1
absolutely: 1
loved: 1
product: 1
worst: 1
service: 1


In [10]:
# Top 5 least frequent words
least_5 = sorted(freq.items(), key=lambda x: x[1])[:5]

print("Top 5 Least Frequent Words:")
for word, count in least_5:
    print(f"{word}: {count}")

Top 5 Least Frequent Words:
get: 1
free: 1
access: 1
absolutely: 1
loved: 1


Due to the small dataset, several tokens have the same frequency of 1, causing some low-frequency words to appear in both the most frequent and least frequent lists.

### Task 6: Build Full Pipeline

In [11]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []
    
    for text in text_list:
        tokens,cleaned = preprocess_text(text)
        clean_sentences.append(cleaned)
        all_tokens.extend(tokens)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

In [12]:
result = full_pipeline(test_sentences)

print("All Tokens:")
print(result["tokens"])

print("\nCleaned Sentences:")
for sentence in result["clean_sentences"]:
    print(sentence)

All Tokens:
['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this']

Cleaned Sentences:
get free access now
absolutely loved this product
worst service ever
call
this the best course
visit now
no this bad
got
win now limited offer
not happy with this


### Task 7: Error Handling

In [13]:
test_cases = [
    "",                # Empty string
    "😍🔥😂",          # Only emojis
    "1234567890",      # Only numbers
]

In [14]:
for sentence in test_cases:
    tokens, cleaned = preprocess_text(sentence)
    print("Original:", sentence)
    print("Tokens:",tokens)
    print("Cleaned Sentence:", cleaned)
    print("*" * 20)

Original: 
Tokens: []
Cleaned Sentence: 
********************
Original: 😍🔥😂
Tokens: []
Cleaned Sentence: 
********************
Original: 1234567890
Tokens: []
Cleaned Sentence: 
********************
